# 00 — Colab setup & smoke test

Clones this repo, installs dependencies, configures keys, and runs a smoke test that
verifies the lightweight `src` modules import and the model registry shapes are sane.

**Keys:**
- `HF_TOKEN` — **required** (gated Qwen / Gemma model downloads + pushing the trained adapter).
- `OPENROUTER_API_KEY` — **optional**, only for Experiment 1 EM%% quantification. The smoke
  test must pass without it.

Convention reminder (enforced project-wide): layer integers are **block indices**
(`_return_layers(model)[i]`), never `output_hidden_states` offsets. Geometry runs in
float32 on CPU; activations are collected in bf16 and cast before any computation.

## 1. Clone the repo

Edit `REPO_URL` to your fork/remote. In Colab this clones fresh each session.

In [ ]:
import os, sys

REPO_URL = "https://github.com/yangdabei/em-persona.git"
REPO_DIR = "/content/em-persona"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR

# If running locally, fall back to the current working tree.
if not os.path.exists(REPO_DIR):
    REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("Repo dir:", REPO_DIR)
print("Contents:", sorted(os.listdir(REPO_DIR)))

## 2. Install dependencies

`unsloth` is only needed for Experiment 1 training; everything else is standard.

In [ ]:
# In Colab, install from requirements.txt. This can take a few minutes.
if IN_COLAB:
    !pip install -q -r requirements.txt
print("Install step done (skipped if local).")

## 3. Configure keys

In Colab, prefer the Secrets panel (key icon) with `HF_TOKEN` and optionally
`OPENROUTER_API_KEY`. The cell below loads them from Colab secrets or the environment.

In [ ]:
import os

def _load_secret(name):
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            os.environ[name] = val
            return val
    except Exception:
        pass
    return os.environ.get(name)

hf = _load_secret("HF_TOKEN")
orouter = _load_secret("OPENROUTER_API_KEY")

assert hf, "HF_TOKEN is required (gated model downloads). Set it in Colab Secrets or env."
print("HF_TOKEN: set")
print("OPENROUTER_API_KEY:", "set (Exp-1 EM%% available)" if orouter else "NOT set (optional; EM%% disabled)")

## 4. Smoke test — lightweight modules

These imports must succeed **without** a GPU and **without** an OpenRouter key.

In [ ]:
from src.data import load_em_eval_prompts
from src.models import MODEL_CONFIGS, assert_model_shape
import src.judges as judges

prompts = load_em_eval_prompts()
print(f"Loaded {len(prompts)} free-form EM eval prompts. Examples:")
for p in prompts[:3]:
    print("  -", p["id"], "::", p["question"][:60])

# Judge harness must degrade gracefully when no key is present.
print("\nOpenRouter judges available:", judges.judges_available())
assert isinstance(judges.judges_available(), bool)  # never raises on missing key

# Registry sanity: the shape asserts that protect Exp1 vs Exp2 tensors.
for k, c in MODEL_CONFIGS.items():
    print(f"  {k}: d={c['hidden_dim']}, n_layers={c['n_layers']}, base={c['base_model']}")
print("\nSmoke test passed.")

## 5. (Optional) GPU + model-access check

Confirms a GPU is present and that the HF token can reach the gated base models. Skip if
you only want to verify the repo wiring.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f"GPU mem: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")

# Cheap auth check: fetch a config (no weights) for the Qwen base model.
from transformers import AutoConfig
cfg = AutoConfig.from_pretrained(MODEL_CONFIGS["qwen-14b"]["base_model"], token=os.environ["HF_TOKEN"])
print("Qwen2.5-14B config OK: hidden_size=", cfg.hidden_size, "layers=", cfg.num_hidden_layers)
assert cfg.hidden_size == 5120 and cfg.num_hidden_layers == 48